In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd

adult = fetch_openml(name="adult", version=2, as_frame=True)
data = adult.frame   # a single DataFrame with all features AND the target column together

data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


# Mandatory Assignment - 2
## Exercise-1: Data Preparation
### Q1.1
 Identify and quantify missing values per feature. Determine whether the missingness looks random or is concentrated in specific subgroups (e.g., check if missing
occupation correlates with workclass). Choose and justify a strategy for handling
them (e.g., imputation vs. removal, and which imputation statistic), and explain
any risks your chosen strategy introduces.

In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


Here we can see that there are some features that are missing some values.

In [14]:
missing = data.isna().sum()
missing_pct = (missing / len(data) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})[missing > 0]

,missing_count,missing_%
workclass,2799,5.73
occupation,2809,5.75
native-country,857,1.75


This shows us that there are some how many that are missing from each the features. 

In [15]:
pd.crosstab(data['workclass'].isna(), data['occupation'].isna())

occupation,False,True
workclass,,
False,46033,10
True,0,2799


By Cross-tabulating missing  `workclass` against missing `occupation`, 
shows us that the all the missing values from `workclass` (2799) are 
also missing in `occupation`. 

My hypothesis is that: individuals with no reported employment status 
naturally have no occupation to report either

A separate, much smaller group of 10 rows has occupation missing despite 
workclass being present, likely reflecting genuine non-response to that 
specific question rather than the same underlying cause.

The best solution to fix this issue is to impute/create a new explicit 
category ("Unknown" or "Not-in-workforce") rather than most-frequent-category 
or removing rows. This will prevent removing valueble information that might 
correlate later with income. I also see for my self that it is bad practice 
assigning hese individuals to whatever category happens to be most common. 

For `native-country` (only 1.75% missing), imputing with the mode 
("United-States", which dominates this column) is reasonable since the
missing fraction is small and unlikely to meaningfully bias results either way.


### Q1.3
Several categorical features are high-cardinality (e.g., native-country, occupation).
Quantify the cardinality of each categorical feature and propose a strategy for the
high-cardinality ones (e.g., grouping rare categories into an ”other” bucket, fre-
quency thresholding). Justify your chosen threshold and show how it changes the
category counts. 

In [16]:
cat_cols = data.select_dtypes(include='category').columns
cardinality = data[cat_cols].nunique().sort_values(ascending=False)
print(cardinality)

native-country    41
education         16
occupation        14
workclass          8
marital-status     7
relationship       6
race               5
sex                2
class              2
dtype: int64


Here can we see how many see how many `native-country` has a high-cardinality 
(Since it has 41 different categories). `Occupation` in the other hand has also 
a high cardinality, but less than the . 

The code bellow will show the categories and the count of each of `native-country` and `Occupation`: 


In [30]:
data['native-country'].value_counts(normalize=True).head(10) * 100

native-country
United-States    91.345212
Mexico            1.981869
Philippines       0.614775
Germany           0.429301
Puerto-Rico       0.383453
Canada            0.379285
El-Salvador       0.323018
India             0.314682
Cuba              0.287590
England           0.264666
Name: proportion, dtype: float64

In [ ]:
data['native-country'].value_counts()

native-country
United-States    91.345212
Mexico            1.981869
Philippines       0.614775
Germany           0.429301
Puerto-Rico       0.383453
Canada            0.379285
El-Salvador       0.323018
India             0.314682
Cuba              0.287590
England           0.264666
Name: proportion, dtype: float64

From this we can see that the US alone accounts for around 90% of all rows,
While mexico has around 1-2%. The rest is under 1%. 

Lets try to reduse it to 3 category. This can be done by creating a threshold 
from 1%:

In [28]:
threshold = 0.01  # keep a category only if it makes up at least 1% of all rows

# get each country's share of the total rows (normalize=True gives proportions, not raw counts)
value_counts = data['native-country'].value_counts(normalize=True)

# countries below the threshold - these are the ones we'll group together
rare_categories = value_counts[value_counts < threshold].index

# create a new column: keep common countries as-is, replace rare ones with "Other"
data['native-country_grouped'] = data['native-country'].apply(
    lambda x: 'Other' if x in rare_categories else x
)


print("Before:", data['native-country'].nunique(), "categories")   # original number of countries
print("After: ", data['native-country_grouped'].nunique(), "categories")  # fewer, now that rare ones are merged
print("\nCategories after grouping:")
print(data['native-country_grouped'].value_counts())

Before: 41 categories
After:  3 categories

Categories after grouping:
native-country_grouped
United-States    43832
Other             3202
Mexico             951
Name: count, dtype: int64



Applying a 1% frequency threshold reduces `native-country` from 41 categories down to
just 3: `United-States` (43,832 rows, around 91%), `Other` (3,202 rows, grouping all
countries individually below 1%), and `Mexico` (951 rows, around 2%).

This removes countries with only a handful of rows each, which the model could barely
learn from anyway, and avoids the risk of a country showing up in the test set that
the model never saw during training.

The trade-off: countries like Germany or Canada, which might have their own real
income pattern, now get lumped into "Other" along with everyone else. Since each of
these groups is very small on its own, we're likely not losing much. But it's worth
knowing this simplification could hide a real pattern if one of these countries
mattered more than expected.